# Chuẩn bị và cập nhật khảo sát tuyến
Biểu mẫu gắn đúng phiên bản của 60 ứng viên và 30 đề xuất **A10/B8/C12**. Tách giá trị OSM, mô hình và quan sát có ngày/bằng chứng. Chưa có dữ liệu thì G1/G4/G5/G6/C6 giữ unknown. Không xây chu trình lái. Hướng dẫn: `work/docs/lez/FIELD_WORKFLOW.md`.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display
candidates = [Path.cwd(), *Path.cwd().parents]
candidates += [p / "work" for p in candidates]
candidates += [Path.home() / "python" / "work"]
WORK = next((p.resolve() for p in candidates if (p / "scripts/lez/project_automation.py").is_file()), None)
if WORK is None:
    raise FileNotFoundError("Mở notebook từ folder work có scripts/lez/project_automation.py.")
if str(WORK / "scripts") not in sys.path:
    sys.path.insert(0, str(WORK / "scripts"))
from lez.common import input_path

from lez import field_workflow as field
from lez import survey_revision

## 1. Tạo hoặc mở chiến dịch hiện hành
Lặp lại cell không xóa ledger quan sát hay dữ liệu chuyến đã nhập. Cache dựa vào snapshot bất biến; thuộc tính cache_hit của báo cáo mới nhất không tạo chiến dịch mới.

In [ ]:
CAMPAIGN = field.prepare(WORK)
DIRECTORY = Path(CAMPAIGN["directory"])
print("Biểu mẫu:", DIRECTORY)
REGISTRY = pd.read_csv(DIRECTORY / "route_registry.csv")
display(REGISTRY)
display(pd.read_csv(DIRECTORY / "desk_review_priority.csv"))

## 2. Kiểm tra trước chuyến
G1: quyền đi đúng nhánh/chiều/ngày/giờ; G4: công trình/phân luồng; G5: lặp ở ít nhất hai ngày; G6: điểm đầu và cuối an toàn. C6 dùng rubric 0–5, cần đủ năm đánh giá thành phần và các G đã qua. Trạng thái LEZ thực thi tại ngày khảo sát được ghi riêng.

In [ ]:
display(pd.read_csv(DIRECTORY / "survey_checklist.csv"))
display(pd.read_csv(DIRECTORY / "evidence_questionnaires.csv"))
print(json.dumps(json.loads((DIRECTORY / "C6_rubric.json").read_text()), ensure_ascii=False, indent=2))
display(pd.read_csv(DIRECTORY / "trip_schedule_template.csv"))

## 3. Nhập quan sát khi có bằng chứng
Xuất CSV theo header `field_observations.csv`, dùng ID/phiên bản từ route_registry; điền `part_id`/`segment_id` khi thuộc tính thay đổi theo đoạn. `coverage=whole_route` chỉ khi kiểm tra toàn tuyến; số làn/tốc độ dùng giá trị đồng nhất trên phạm vi đó. Bằng chứng bàn giấy không trở thành quan sát thực địa.
Đặt `OBSERVATIONS_CSV` tới tệp đã điền. Mặc định None; không tạo quan sát giả.

In [ ]:
OBSERVATIONS_CSV = None  # ví dụ: WORK / "data/measurements/incoming/observations.csv"
if OBSERVATIONS_CSV is not None:
    IMPORT = field.import_observations(DIRECTORY, Path(OBSERVATIONS_CSV))
    print(json.dumps(IMPORT, ensure_ascii=False, indent=2))
    if not IMPORT["valid"]:
        raise ValueError("Sửa các lỗi import được liệt kê; batch lỗi chưa ghi vào ledger.")
else:
    print("Chờ CSV quan sát có bằng chứng.")

## 4. Đánh giá bằng chứng và dữ liệu đã có
Mỗi quan sát giữ ngày, người ghi, nguồn, đơn vị và hash tuyến. CSV cũ hoặc sai phiên bản bị từ chối. Import lặp đúng ID được bỏ qua; không ghi đè bằng chứng cũ.

In [ ]:
STATE = field.status(DIRECTORY)
print(json.dumps(STATE, ensure_ascii=False, indent=2))
display(pd.read_csv(DIRECTORY / "proposed_decisions.csv"))
display(pd.read_csv(DIRECTORY / "attributes_osm.csv"))
display(pd.read_csv(DIRECTORY / "attributes_models.csv"))

## 5. Cập nhật thuộc tính, chấm lại và thay tuyến
Làn/tốc độ trên đoạn quan sát được tính theo chiều dài thực đã đi; điểm đơn lẻ và số nút trên một phần tuyến không được suy cho toàn tuyến. C6 đủ chứng cứ thay giả định 2,5; C1–C7 và chuẩn hóa X được tính lại bằng thuật toán cũ. Tuyến lỗi G, C6=0 hoặc xung đột graph bị loại khỏi tập khả dụng.
Tính lại toàn bộ ma trận, D, độ phủ, trùng hành lang, đa dạng và quota 10/8/12. Nếu không còn đủ ứng viên, báo infeasible, không tự nới quota. Chỉ công bố snapshot khảo sát kỹ thuật khi cả 30 có đủ chứng cứ; snapshot đó không xác nhận GIS pháp lý.

In [ ]:
PLAN = field.build_update_plan(DIRECTORY, evaluate_reserves=True)
REVISION = survey_revision.run(DIRECTORY, WORK)
print(json.dumps(REVISION, ensure_ascii=False, indent=2))
display(pd.read_csv(DIRECTORY / "reserve_replacement_dry_run.csv"))
if REVISION.get("paths", {}).get("directory"):
    revised = input_path(WORK, REVISION["paths"]["directory"])
    display(pd.read_csv(revised / "field_feature_overlay.csv"))

## 6. Chuyển sang kiểm tra dữ liệu đo
Nhập `trips.csv` và `gps_points.csv` theo biểu mẫu ở `work/data/measurements/incoming`; dùng đúng route_version/source_bundle_sha256 và timestamp có múi giờ Hà Nội. Chạy notebook 08 hoặc bộ điều phối notebook 09. Lịch 180 lượt trong biểu mẫu là kế hoạch, chưa phải chuyến đã đo.